---
title: GSB545 Advanced Machine Learning 04/07/2026 In Class Coding
author: Alexa Dandridge
format:
    html:
        embed-resources: true
        code-line numbers: true

---

In [20]:
# importing libraries
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

In [21]:
# loading the dataset and initial EDU with SweetViz
insurance = pd.read_csv('insurance.csv')
insurance.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [22]:
# visualizing the data with SweetViz (This is the EDA)
report = sv.analyze(insurance)
report.show_html('insurance_report.html')
# This pops out an html file with the report.

                                             |          | [  0%]   00:00 -> (? left)

Report insurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [23]:
# encoding categorical variables
insurance_encoded = pd.get_dummies(insurance, drop_first=True).astype(int)
insurance_encoded.head()

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0


Describing the trends in the data that I see after running a quick EDA: 

The dataset includes variables such as age, bmi, number of children, smoker status, region, and charges, with no missing values. The distribution of age is fairly even, but we see that there is a higher amount of younger people (close to 18/19) in the data. Sex is split pretty evenly too, showing that there is not a substantial imbalance in gender representation. Smoker status is highly imbalanced, with most people being non-smokers. Region is fairly evenly distributed also. Charges is skewed right, indicating that there may be some factors that increase costs. 

In [24]:
# In this cell, I am splittng the data into features and target variable, then into training and testing sets, and finally scaling the features for modeling.
# preparing the data for modeling
X = insurance_encoded.drop('charges', axis=1)
y = insurance_encoded['charges']

# splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scaling the features (don't need for RF but do for linear regression)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [25]:
# baselinelinear regression model
baseline_lr = LinearRegression()
baseline_lr.fit(X_train_scaled, y_train)
y_pred_lr = baseline_lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

# baseline random forest model
baseline_rf = RandomForestRegressor(random_state=42)
baseline_rf.fit(X_train, y_train)
y_pred_rf = baseline_rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

# print baseline results
print(f'Baseline Linear Regression MSE and R2: {mse_lr:.2f}, R2: {r2_lr:.2f}')
print(f'Baseline Random Forest MSE and R2: {mse_rf:.2f}, R2: {r2_rf:.2f}')

Baseline Linear Regression MSE and R2: 33566439.74, R2: 0.78
Baseline Random Forest MSE and R2: 22179121.67, R2: 0.86


The Random Forest Regression model outperformed the Linear Regression model because it has a higher R2 value and a lower mean squared error value. This suggests that the Random Forest model is better able to provide predictive accuracy and capture teh relationships between the variables in the model.

In [26]:

# CV with baseline models with r2 as the scoring metric
cv_scores_lr = cross_val_score(baseline_lr, X_train_scaled, y_train, cv=5, scoring='r2')
cv_scores_rf = cross_val_score(baseline_rf, X_train, y_train, cv=5, scoring='r2')
print(f'Baseline Linear Regression CV R2 Scores: {cv_scores_lr}')
print(f'Baseline Random Forest CV R2 Scores: {cv_scores_rf}') 

Baseline Linear Regression CV R2 Scores: [0.71508701 0.8018118  0.72333011 0.65748491 0.76763199]
Baseline Random Forest CV R2 Scores: [0.81530553 0.89851976 0.79006114 0.78263259 0.82825938]


In [27]:
# gridsearchCV for hyperparameter tuning of random forest
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'max_features': ['log2', 'sqrt']
}           

grid_search_rf = GridSearchCV(estimator=baseline_rf, 
                              param_grid=param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)    
print(f'Best RF Hyperparameters: {grid_search_rf.best_params_}')
print(f'Best RF CV R2 Score: {grid_search_rf.best_score_:.2f}')

Best RF Hyperparameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
Best RF CV R2 Score: 0.84


The best set of hyperparameters from your search include: Best RF Hyperparameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}.

In [28]:
# examine top 20 ranked parameter combinations from grid search 
print("\nTop 20 GridSearchCV results:")
top20_eval = (
    pd.DataFrame(grid_search_rf.cv_results_)
    .sort_values('mean_test_score', ascending=False)
    .head(20)
    [['param_n_estimators', 'param_max_depth', 'param_min_samples_split', 
      'param_max_features', 'mean_test_score']]
    .rename(columns=lambda x: x.replace('param_', ''))
)
print(top20_eval)


Top 20 GridSearchCV results:
    n_estimators max_depth  min_samples_split max_features  mean_test_score
11           200        10                  5         log2         0.841202
3            200      None                  5         log2         0.840662
19           200        20                  5         log2         0.840662
10           100        10                  5         log2         0.840452
8            100        10                  2         log2         0.840383
9            200        10                  2         log2         0.839967
2            100      None                  5         log2         0.839195
18           100        20                  5         log2         0.839195
17           200        20                  2         log2         0.836639
1            200      None                  2         log2         0.836593
16           100        20                  2         log2         0.835721
0            100      None                  2         log2

I would choose the hyperparameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}. This is because they provide the highest performing model when it comes to the R2 score. 
It provides the highest cross-val score without making the model too extremely complex but using these hyperparameters. 

In [29]:
# Best model from GridSearchCV
best_rf = grid_search_rf.best_estimator_

# Predict on training and test data
y_train_pred = best_rf.predict(X_train)
y_test_pred = best_rf.predict(X_test)

# Calculate training performance
train_r2 = r2_score(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)

# Calculate test performance
test_r2 = r2_score(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

# Print results
print("Training R2:", round(train_r2, 4))
print("Training MSE:", round(train_mse, 2))
print("Test R2:", round(test_r2, 4))
print("Test MSE:", round(test_mse, 2))

Training R2: 0.9302
Training MSE: 10067783.72
Test R2: 0.8684
Test MSE: 20430805.91


The model performs well with both training and test data because the training R2 is 0.93 and test R2 is 0.87. While there is a difference, it is small (0.06), which indicates slight overfitting, but still strong predictive ability by the model. 